### Library and Dataset imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
sns.set_style('whitegrid')

In [ ]:
print("Downloading dataset...")
path = kagglehub.dataset_download("redwankarimsony/heart-disease-data")

# Load the dataset from the downloaded path
file_path = f'{path}/heart_disease_uci.csv'
df = pd.read_csv(file_path)

print("Dataset downloaded and loaded successfully.")
print(f"Data shape: {df.shape}")
df.head()

## EDA

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

### Target Variable analysis

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='num', data=df, hue='num')
plt.title('Distribution of Heart Disease (1 = Disease, 0 = No Disease)')
plt.xlabel('Target')
plt.ylabel('Count')
plt.show()

In [ ]:
df['num'].value_counts()

**Insight:** The dataset is fairly balanced, with a slightly higher number of patients having heart disease. This is good because it means our model will have a similar number of examples for both classes to learn from, and accuracy will be a meaningful metric.

### Feature vs Target analysis

In [ ]:
# Let's visualize the relationship between key features and the target
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Key Features vs. Heart Disease', fontsize=16)

# Age vs. Target
sns.histplot(ax=axes[0, 0], data=df, x='age', hue='num', multiple='stack', palette='plasma').set_title('Age Distribution by Target')

# Max Heart Rate(thalch) vs. Target
sns.boxplot(ax=axes[0, 1], data=df, x='num', y='thalch', palette='magma', hue='num', legend=False).set_title('Max Heart Rate by Target')

# Chest Pain Type(cp) vs. Target
cp_plot = sns.countplot(ax=axes[1, 0], data=df, x='cp', hue='num', palette='cividis')
cp_plot.set_title('Chest Pain Type by Target')
cp_plot.set_xticks(range(len(df['cp'].unique())))
cp_plot.set_xticklabels(['Typical Angina', 'Atypical Angina', 'Non-anginal Pain', 'Asymptomatic'])

# Sex vs. Target
sex_plot = sns.countplot(ax=axes[1, 1], data=df, x='sex', hue='num', palette='inferno')
sex_plot.set_title('Sex by Target')
sex_plot.set_xticks(range(len(df['sex'].unique())))
sex_plot.set_xticklabels(['Female', 'Male'])

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(data=df, x='chol', hue='num', multiple='stack', palette='plasma').set_title('Age Distribution by Target')

plt.show()


In [ ]:
# Correlation Heatmap
plt.figure(figsize=(16, 12))
# Select only numerical columns for correlation calculation
numerical_df = df.select_dtypes(include=np.number)
sns.heatmap(numerical_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.show()

### ca has high correlation with num

## Data Preprocessing

### Separating the target from features.

In [ ]:
print(X)

In [ ]:
X = df.drop(['id', 'dataset', 'num'], axis=1)
print(X)

In [ ]:
y = df['num']
print(y)

In [ ]:
df.isna().sum()

trestbps, chol, thalch, oldpeak, ca - numerical feature

In [ ]:
trestbps_mean = round(df['trestbps'].mean(),1)
df['trestbps'] = df['trestbps'].fillna(trestbps_mean)

chol_mean = round(df['chol'].mean(),1)
df['chol'] = df['chol'].fillna(chol_mean)

thalch_mean = round(df['thalch'].mean(),1)
df['thalch'] = df['thalch'].fillna(thalch_mean)

oldpeak_mean = round(df['oldpeak'].mean(),1)
df['oldpeak'] = df['oldpeak'].fillna(oldpeak_mean)

ca_mean = round(df['ca'].mean(),1)
df['ca'] = df['ca'].fillna(ca_mean)

In [ ]:
fbs_mode = df['fbs'].mode()[0]
df['fbs'] = df['fbs'].fillna(fbs_mode)

restecg_mode = df['restecg'].mode()[0]
df['restecg'] = df['restecg'].fillna(restecg_mode)

exang_mode = df['exang'].mode()[0]
df['exang'] = df['exang'].fillna(exang_mode)

slope_mode = df['slope'].mode()[0]
df['slope'] = df['slope'].fillna(slope_mode)

thal_mode = df['thal'].mode()[0]
df['thal'] = df['thal'].fillna(thal_mode)

In [ ]:
df.isnull().sum()

In [ ]:
print(df.head())

## Model Buildnig & Training

In [ ]:
X = df.drop(['id', 'dataset', 'num'], axis=1)
y = df['num']

In [ ]:
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal', 'ca']

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print(X_encoded.head())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', C=1.0)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate_model(name, y_test, y_pred):
    print(f"\n🧠 {name} Results")
    print("-" * 30)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

# Evaluate all models
evaluate_model("Logistic Regression", y_test, y_pred_lr)
evaluate_model("Random Forest", y_test, y_pred_rf)
evaluate_model("SVM", y_test, y_pred_svm)
evaluate_model("KNN", y_test, y_pred_knn)

## Conclusion-

SVM has better accuracy, slight better F1 score.

In [ ]:
cm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Disease', 'Severity 2', 'Severity 3', 'Severity 4'], yticklabels=['No Disease', 'Disease', 'Severity 2', 'Severity 3', 'Severity 4'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Support Vector Machine (SVM)')
plt.show()

SVM	Best performer overall — high accuracy and balanced classification across classes. Excellent at finding complex patterns, especially with scaled data.
Random Forest	Performed well, especially on categorical data. Didn’t need scaling. May have been slightly less accurate than SVM but still strong.
Logistic Regression	Simple, interpretable, but may struggle with complex or non-linear patterns. Decent baseline model.
KNN	Performance depends heavily on feature scaling and value of k. Can struggle with large feature spaces.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Get feature importances
importances = rf.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Plot top features
plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(10))
plt.title("Top 10 Feature Importances - Random Forest")
plt.show()

**Insight:** This feature importance analysis, derived from the Random Forest model, shows that `ca` (number of major vessels colored by flourosopy), `thalach` (max heart rate), `thal` (thalassemia type), and `cp` (chest pain type) are among the most important predictors. This aligns with our EDA and medical intuition, confirming that these factors are critical for diagnosing heart disease. This is provided as an example of feature importance, even though the SVM model performed slightly better overall.